# B2 — Frozen multilingual sentence encoder + Logistic Regression

Linear probe on top of `paraphrase-multilingual-MiniLM-L12-v2` (frozen). 384-dim embeddings.

## Imports and data loading

In [1]:
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, mean_absolute_error

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 128

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/ismaillataoui/miniconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/ismaillataoui/miniconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/Users/ismaillataoui/CompIntelLab/Project/code/.venv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/ismaillataoui/CompIntelLab/Project/code/.venv/lib/python3.9/site-packages/traitlets/confi

## Reuse the canonical val split from B1

Same indices → directly comparable scores.

In [2]:
val_idx = np.load('data/val_indices.npy')
train_df = train.drop(index=val_idx).reset_index(drop=True)
val_df   = train.loc[val_idx].reset_index(drop=True)
y_train = train_df['label'].values
y_val   = val_df['label'].values

## Load encoder

Uses MPS on Apple Silicon, CUDA on the cluster, CPU as fallback.

In [3]:
device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
encoder = SentenceTransformer(MODEL_NAME, device=device)
print('device:', device, ' dim:', encoder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

device: cpu  dim: 384


## Encode train / val / test

Cached to `data/b2_*.npy` — if the file exists, we skip the encode.

In [4]:
def encode_cached(texts, cache_path):
    if os.path.exists(cache_path):
        return np.load(cache_path)
    emb = encoder.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(cache_path, emb)
    return emb

X_train = encode_cached(train_df['sentence'].tolist(), 'data/b2_train.npy')
X_val   = encode_cached(val_df['sentence'].tolist(),   'data/b2_val.npy')
X_test  = encode_cached(test['sentence'].tolist(),     'data/b2_test.npy')
print(X_train.shape, X_val.shape, X_test.shape)

Batches:   0%|          | 0/1772 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Train logistic regression

In [ ]:
clf = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

## Evaluate — train and val

In [ ]:
def report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    score = 1 - mae / 4
    acc = float(np.mean(y_true == y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    print(f'[{name}] score={score:.4f}  MAE={mae:.4f}  acc={acc:.4f}')
    print(cm)
    return score

report(y_train, clf.predict(X_train), 'train')
report(y_val,   clf.predict(X_val),   'val')

## MAE-optimal rounding

In [ ]:
def median_round(probs):
    cum = np.cumsum(probs, axis=1)
    return np.argmax(cum >= 0.5, axis=1)

report(y_val, median_round(clf.predict_proba(X_val)), 'val_med')

## Write Kaggle submission

In [ ]:
os.makedirs('submissions', exist_ok=True)
test_pred = median_round(clf.predict_proba(X_test))
pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)}).to_csv(
    'submissions/b2_sentemb_logreg.csv', index=False,
)